# Análisis de MSCI World

Este notebook contiene el análisis exploratorio de los datos históricos del índice MSCI World.


In [5]:
from warnings import filterwarnings
filterwarnings("ignore")


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [7]:
import os

os.getcwd()

'/home/agusriscos/proyectos/ARPTools'

## 1. Carga de Datos


In [8]:
# Obtener la ruta del proyecto
proyecto_root = Path().resolve().parent.parent.parent
archivo_parquet = "data/16dic25/msci_world.parquet"

# Cargar datos
df = pd.read_parquet(archivo_parquet, engine="pyarrow")
print(f"Datos cargados desde: {archivo_parquet}")
print(f"Shape del DataFrame: {df.shape}")


Datos cargados desde: data/16dic25/msci_world.parquet
Shape del DataFrame: (3490, 8)


## 2. Exploración Inicial


In [9]:
# Información general del DataFrame
df.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3490 entries, 2012-01-12 00:00:00-05:00 to 2025-11-26 00:00:00-05:00
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           3490 non-null   float64
 1   High           3490 non-null   float64
 2   Low            3490 non-null   float64
 3   Close          3490 non-null   float64
 4   Volume         3490 non-null   int64  
 5   Dividends      3490 non-null   float64
 6   Stock Splits   3490 non-null   float64
 7   Capital Gains  3490 non-null   float64
dtypes: float64(7), int64(1)
memory usage: 245.4 KB


In [10]:
# Primeras filas
df.head(10)


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2012-01-12 00:00:00-05:00,38.786598,38.786598,38.786598,38.786598,100,0.0,0.0,0.0
2012-01-13 00:00:00-05:00,38.786598,38.786598,38.786598,38.786598,0,0.0,0.0,0.0
2012-01-17 00:00:00-05:00,38.786598,38.786598,38.786598,38.786598,0,0.0,0.0,0.0
2012-01-18 00:00:00-05:00,38.786598,38.786598,38.786598,38.786598,0,0.0,0.0,0.0
2012-01-19 00:00:00-05:00,39.927841,39.927841,39.927841,39.927841,300,0.0,0.0,0.0
2012-01-20 00:00:00-05:00,39.927841,39.927841,39.927841,39.927841,0,0.0,0.0,0.0
2012-01-23 00:00:00-05:00,39.927841,39.927841,39.927841,39.927841,0,0.0,0.0,0.0
2012-01-24 00:00:00-05:00,39.827587,39.827587,39.827587,39.827587,400,0.0,0.0,0.0
2012-01-25 00:00:00-05:00,39.827587,39.827587,39.827587,39.827587,0,0.0,0.0,0.0


In [11]:
# Últimas filas
df.tail(10)


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2025-11-13 00:00:00-05:00,185.529999,185.589996,182.940002,183.229996,320900,0.0,0.0,0.0
2025-11-14 00:00:00-05:00,181.679993,184.020004,181.199997,183.190002,233700,0.0,0.0,0.0
2025-11-17 00:00:00-05:00,182.399994,183.369995,180.449997,181.279999,496600,0.0,0.0,0.0
2025-11-18 00:00:00-05:00,180.020004,180.779999,178.570007,179.770004,229200,0.0,0.0,0.0
2025-11-19 00:00:00-05:00,179.750000,181.179993,179.119995,180.059998,208700,0.0,0.0,0.0
2025-11-20 00:00:00-05:00,182.470001,183.080002,177.229996,177.300003,396800,0.0,0.0,0.0
2025-11-21 00:00:00-05:00,178.229996,180.440002,177.160004,179.179993,893400,0.0,0.0,0.0
2025-11-24 00:00:00-05:00,179.979996,181.570007,179.610001,181.160004,886000,0.0,0.0,0.0
2025-11-25 00:00:00-05:00,181.470001,183.419998,180.490005,183.220001,820900,0.0,0.0,0.0


In [12]:
# Estadísticas descriptivas
df.describe()


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
count,3490.000000,3490.000000,3490.000000,3490.000000,3.490000e+03,3490.000000,3490.0,3490.0
mean,88.366346,88.754059,87.903486,88.347561,1.183058e+05,0.006903,0.0,0.0
std,36.058105,36.230088,35.873279,36.075737,1.778967e+05,0.080116,0.0,0.0
min,38.215985,38.215985,38.030919,38.038631,0.000000e+00,0.000000,0.0,0.0
25%,59.160574,59.341564,58.849710,59.081757,9.600000e+03,0.000000,0.0,0.0
50%,78.659188,78.897634,78.296035,78.513264,4.520000e+04,0.000000,0.0,0.0
75%,114.578853,115.078395,114.001836,114.721077,1.697500e+05,0.000000,0.0,0.0
max,186.880005,187.070007,186.110001,186.639999,3.485200e+06,1.261000,0.0,0.0


In [13]:
# Rango de fechas
fecha_min = pd.Timestamp(df.index.min())
fecha_max = pd.Timestamp(df.index.max())
diferencia = fecha_max - fecha_min
print(f"Fecha mínima: {fecha_min}")
print(f"Fecha máxima: {fecha_max}")
print(f"Total de días: {len(df)}")
if hasattr(diferencia, 'days'):
    print(f"Años cubiertos: {diferencia.days / 365.25:.2f}")
else:
    print(f"Años cubiertos: {diferencia.total_seconds() / (365.25 * 24 * 3600):.2f}")


Fecha mínima: 2012-01-12 00:00:00-05:00
Fecha máxima: 2025-11-26 00:00:00-05:00
Total de días: 3490
Años cubiertos: 13.87


In [14]:
# Valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())
print(f"\nPorcentaje de valores faltantes:")
print((df.isnull().sum() / len(df) * 100).round(2))


Valores faltantes por columna:
Open             0
High             0
Low              0
Close            0
Volume           0
Dividends        0
Stock Splits     0
Capital Gains    0
dtype: int64

Porcentaje de valores faltantes:
Open             0.0
High             0.0
Low              0.0
Close            0.0
Volume           0.0
Dividends        0.0
Stock Splits     0.0
Capital Gains    0.0
dtype: float64


## 3. Análisis de Precios


In [15]:
# Visualización de precios de cierre
fig = px.line(
    df, 
    y="Close",
    title="Evolución del Precio de Cierre - MSCI World",
    labels={"Date": "Fecha", "Close": "Precio de Cierre"}
)
fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Precio",
    hovermode="x unified"
)
fig.show()


In [25]:
# Visualization of returns (English version - no subplot titles)
# Calculate required columns if they don't exist
if "Returns" not in df.columns:
    df["Returns"] = df["Close"].pct_change() * 100

if "Volatility_30d" not in df.columns:
    df["Volatility_30d"] = df["Returns"].rolling(window=30).std()

if "MA_50" not in df.columns:
    df["MA_50"] = df["Close"].rolling(window=50).mean()

if "MA_200" not in df.columns:
    df["MA_200"] = df["Close"].rolling(window=200).mean()

fig = make_subplots(
    rows=3, cols=1,
    vertical_spacing=0.08,
    row_heights=[0.4, 0.3, 0.3]
)

# Price with moving averages
fig.add_trace(
    go.Scatter(x=df.index, y=df["Close"], name="Close", line=dict(color="blue")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df["MA_50"], name="MA 50", line=dict(color="orange", dash="dash")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df["MA_200"], name="MA 200", line=dict(color="red", dash="dash")),
    row=1, col=1
)

# Returns
fig.add_trace(
    go.Scatter(x=df.index, y=df["Returns"], name="Returns", line=dict(color="green"), mode="lines"),
    row=2, col=1
)
# Horizontal line at y=0 for returns
fig.add_shape(
    type="line",
    x0=df.index.min(),
    x1=df.index.max(),
    y0=0,
    y1=0,
    line=dict(dash="dash", color="black", width=1),
    opacity=0.3,
    row=2,
    col=1
)

# Volatility
fig.add_trace(
    go.Scatter(x=df.index, y=df["Volatility_30d"], name="30-day Volatility", line=dict(color="purple"), fill="tozeroy"),
    row=3, col=1
)

fig.update_layout(
    title="Price, Returns and Volatility Analysis - MSCI World",
    height=900,
    showlegend=True
)

fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Returns (%)", row=2, col=1)
fig.update_yaxes(title_text="Volatility", row=3, col=1)

fig.show()

In [26]:
# Visualización de OHLC (Open, High, Low, Close)
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Precios OHLC", "Volumen"),
    vertical_spacing=0.1,
    row_heights=[0.7, 0.3]
)

# Gráfico de velas (candlestick)
fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["Open"],
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        name="OHLC"
    ),
    row=1, col=1
)

# Gráfico de volumen
fig.add_trace(
    go.Bar(x=df.index, y=df["Volume"], name="Volumen", marker_color="blue"),
    row=2, col=1
)

fig.update_layout(
    title="Análisis OHLC y Volumen - MSCI World",
    xaxis_title="Fecha",
    height=800,
    showlegend=True
)

fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_yaxes(title_text="Precio", row=1, col=1)
fig.update_yaxes(title_text="Volumen", row=2, col=1)

fig.show()


## 4. Cálculo de Returns


In [18]:
# Calcular returns diarios (porcentaje)
df["Returns"] = df["Close"].pct_change() * 100

# Calcular returns logarítmicos
df["Log_Returns"] = np.log(df["Close"] / df["Close"].shift(1)) * 100

# Calcular volatilidad móvil (rolling 30 días)
df["Volatility_30d"] = df["Returns"].rolling(window=30).std()

# Calcular media móvil de 50 y 200 días
df["MA_50"] = df["Close"].rolling(window=50).mean()
df["MA_200"] = df["Close"].rolling(window=200).mean()

print("Métricas calculadas:")
print("- Returns (diarios, %): Calculado")
print("- Log Returns (diarios, %): Calculado")
print("- Volatilidad móvil (30 días): Calculado")
print("- Media móvil 50 días: Calculado")
print("- Media móvil 200 días: Calculado")


Métricas calculadas:
- Returns (diarios, %): Calculado
- Log Returns (diarios, %): Calculado
- Volatilidad móvil (30 días): Calculado
- Media móvil 50 días: Calculado
- Media móvil 200 días: Calculado


In [19]:
# Estadísticas de returns
print("Estadísticas de Returns Diarios (%):")
print(df["Returns"].describe())
print(f"\nSkewness: {df['Returns'].skew():.4f}")
print(f"Kurtosis: {df['Returns'].kurtosis():.4f}")


Estadísticas de Returns Diarios (%):
count    3489.000000
mean        0.050577
std         1.077997
min       -11.377599
25%        -0.387660
50%         0.028133
75%         0.539670
max         9.096323
Name: Returns, dtype: float64

Skewness: -0.4719
Kurtosis: 13.6188


In [20]:
# Visualización de returns
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Precio de Cierre con Medias Móviles", "Returns Diarios (%)", "Volatilidad Móvil (30 días)"),
    vertical_spacing=0.08,
    row_heights=[0.4, 0.3, 0.3]
)

# Precio con medias móviles
fig.add_trace(
    go.Scatter(x=df.index, y=df["Close"], name="Close", line=dict(color="blue")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df["MA_50"], name="MA 50", line=dict(color="orange", dash="dash")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df["MA_200"], name="MA 200", line=dict(color="red", dash="dash")),
    row=1, col=1
)

# Returns
fig.add_trace(
    go.Scatter(x=df.index, y=df["Returns"], name="Returns", line=dict(color="green"), mode="lines"),
    row=2, col=1
)
# Línea horizontal en y=0 para returns
fig.add_shape(
    type="line",
    x0=df.index.min(),
    x1=df.index.max(),
    y0=0,
    y1=0,
    line=dict(dash="dash", color="black", width=1),
    opacity=0.3,
    row=2,
    col=1
)

# Volatilidad
fig.add_trace(
    go.Scatter(x=df.index, y=df["Volatility_30d"], name="Volatilidad 30d", line=dict(color="purple"), fill="tozeroy"),
    row=3, col=1
)

fig.update_layout(
    title="Análisis de Precios, Returns y Volatilidad - MSCI World",
    height=900,
    showlegend=True
)

fig.update_xaxes(title_text="Fecha", row=3, col=1)
fig.update_yaxes(title_text="Precio", row=1, col=1)
fig.update_yaxes(title_text="Returns (%)", row=2, col=1)
fig.update_yaxes(title_text="Volatilidad", row=3, col=1)

fig.show()


In [21]:
# Distribución de returns
fig = px.histogram(
    df.dropna(subset=["Returns"]),
    x="Returns",
    nbins=100,
    title="Distribución de Returns Diarios (%)",
    labels={"Returns": "Returns (%)", "count": "Frecuencia"}
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Media")
fig.update_layout(
    xaxis_title="Returns (%)",
    yaxis_title="Frecuencia"
)
fig.show()


## 5. Análisis de Dividends y Stock Splits


In [22]:
# Análisis de dividends
print("Análisis de Dividends:")
print(f"Total de días con dividendos: {(df['Dividends'] > 0).sum()}")
print(f"Total de dividendos pagados: {df['Dividends'].sum():.4f}")
print(f"Dividendo promedio (cuando > 0): {df[df['Dividends'] > 0]['Dividends'].mean():.4f}")
print(f"Dividendo máximo: {df['Dividends'].max():.4f}")

# Análisis de stock splits
print("\nAnálisis de Stock Splits:")
print(f"Total de días con stock splits: {(df['Stock Splits'] > 0).sum()}")
if (df['Stock Splits'] > 0).sum() > 0:
    print(f"Stock splits únicos: {df[df['Stock Splits'] > 0]['Stock Splits'].unique()}")


Análisis de Dividends:
Total de días con dividendos: 29
Total de dividendos pagados: 24.0930
Dividendo promedio (cuando > 0): 0.8308
Dividendo máximo: 1.2610

Análisis de Stock Splits:
Total de días con stock splits: 0


## 6. Análisis por Períodos


In [23]:
# Análisis por año
df["Year"] = df.index.year
returns_anuales = df.groupby("Year")["Returns"].agg(["mean", "std", "sum"]).round(4)
returns_anuales.columns = ["Media_Returns", "Volatilidad", "Return_Total"]
print("Estadísticas de Returns por Año:")
print(returns_anuales)


Estadísticas de Returns por Año:
      Media_Returns  Volatilidad  Return_Total
Year                                          
2012         0.0603       0.8467       14.5816
2013         0.1027       1.3333       25.8745
2014         0.0201       0.7944        5.0715
2015         0.0027       1.0225        0.6772
2016         0.0327       0.9654        8.2293
2017         0.0832       0.4171       20.8836
2018        -0.0310       0.9606       -7.7908
2019         0.1011       0.7290       25.4805
2020         0.0796       2.0685       20.1291
2021         0.0829       0.7802       20.8814
2022        -0.0682       1.4636      -17.1218
2023         0.0891       0.8010       22.2764
2024         0.0708       0.7564       17.8346
2025         0.0857       1.1389       19.4545


In [24]:
# Visualización de returns anuales
fig = px.bar(
    returns_anuales.reset_index(),
    x="Year",
    y="Return_Total",
    title="Returns Totales por Año (%)",
    labels={"Year": "Año", "Return_Total": "Return Total (%)"}
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.update_layout(
    xaxis_title="Año",
    yaxis_title="Return Total (%)"
)
fig.show()
